# 00 — Setup: import project code + train/val data into this Studio space
Runs on the JupyterLab GPU app (ml.g6e.2xlarge, 1× NVIDIA L40S 48 GB). **Only the train/val archive is copied. The test archive (`data/test_holdout/`) must NOT be downloaded** — the protocol forbids it until the method is frozen and the user approves, even though this Studio role can technically read it.

**Single project root:** `~/brain-mri-segmentation/` → `notebooks/ src/ scripts/ configs/ tests/ sagemaker/ data/ (train+val only) runs/ evidence/ docs/ data_manifests/ README.md`, downloads cached in `.downloads/`. Nothing is deleted; existing files with the same name are overwritten only inside this root.

**Data provenance (disclose in report):** the authoritative dataset is the course package in Google Drive (`LGG-MRI-Segmentation/dataset`, fixed patient split 77/17/16, 2,763/598/568 pairs). The staged TIFF bytes were obtained from a public Hugging Face copy of the Kaggle dataset (`Ehsan-rmz/lgg-mri-segmentation-research`), with one file re-fetched from Drive; **every one of the 7,858 files was verified byte-identical to the course manifest SHA-256**, and split membership comes only from the course manifests. Images keep their 3 stored channels (pre-contrast, FLAIR, post-contrast slots); masks are binarised with `mask > 0`.

In [1]:
import os, subprocess
HOME=os.path.expanduser('~'); ROOT=os.path.realpath(os.path.expanduser('~/brain-mri-segmentation')); os.makedirs(ROOT,exist_ok=True); B='ajay-mlp1-lgg-044450716277-use1'
import shutil
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout if shutil.which('nvidia-smi') else 'nvidia-smi not found (no GPU?)')

Sun Nov  8 09:06:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.112.07             Driver Version: 595.112.07     CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:30:00.0 Off |                    0 |
| N/A   32C    P0             66W /  350W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch; print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'available', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

torch 2.8.0 CUDA 12.9 available True
NVIDIA L40S


In [3]:
%pip install -q tifffile pyyaml pandas

Note: you may need to restart the kernel to use updated packages.


## Copy project source (from S3 `studio/`)

In [4]:
import tarfile, subprocess
ROOT=os.path.realpath(os.path.expanduser('~/brain-mri-segmentation')); os.makedirs(ROOT,exist_ok=True)
for d in ['notebooks','data','runs','evidence','docs']: os.makedirs(f'{ROOT}/{d}',exist_ok=True)
SRC_TAR=f'{ROOT}/.downloads/ml-project1-src.tar.gz'; os.makedirs(os.path.dirname(SRC_TAR),exist_ok=True)
subprocess.run(['aws','s3','cp',f's3://{B}/studio/v2/ml-project1-src.tar.gz',SRC_TAR],check=True)
ALLOWED={'src','scripts','configs','tests','sagemaker','requirements.txt','requirements-gpu.txt','README.md','docs','evidence','data_manifests'}
with tarfile.open(SRC_TAR) as t:
    keep=[]
    for m in t.getmembers():
        name=m.name[2:] if m.name.startswith('./') else m.name
        if not name or name=='.': continue
        top=name.split('/')[0]
        if top=='notebooks': continue  # keep the v4 notebooks already in ROOT/notebooks
        assert top in ALLOWED, f'unexpected member {m.name}'
        assert m.isfile() or m.isdir(), f'unexpected type {m.name}'
        tgt=os.path.realpath(os.path.join(ROOT,name)); assert tgt.startswith(ROOT+os.sep), f'unsafe {m.name}'
        keep.append(m)
    t.extractall(ROOT,members=keep)
print('source extracted into',ROOT); print(sorted(os.listdir(ROOT)))

download: s3://ajay-mlp1-lgg-044450716277-use1/studio/v2/ml-project1-src.tar.gz to ../.downloads/ml-project1-src.tar.gz
source extracted into /home/sagemaker-user/brain-mri-segmentation
['.downloads', 'README.md', 'configs', 'data', 'data_manifests', 'docs', 'evidence', 'notebooks', 'requirements-gpu.txt', 'requirements.txt', 'runs', 'sagemaker', 'scripts', 'src', 'tests']


/tmp/ipykernel_808/2270601602.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(ROOT,members=keep)


## Copy TRAIN/VAL data only and verify

In [5]:
import hashlib, tarfile, subprocess
TAR=f'{ROOT}/.downloads/trainval.tar'; EXPECTED='d20e09fe1930086d6c913a73206038abca362d98c12f5e68ee6b3d3e921ad0ae'
subprocess.run(['aws','s3','cp',f's3://{B}/data/trainval/trainval.tar',TAR],check=True)
h=hashlib.sha256(open(TAR,'rb').read()).hexdigest()
assert h==EXPECTED, f'trainval.tar sha256 mismatch: {h}'
DEST=os.path.realpath(f'{ROOT}/data'); os.makedirs(DEST,exist_ok=True)
with tarfile.open(TAR) as t:
    names=t.getnames()
    for mbr in t.getmembers():
        target=os.path.realpath(os.path.join(DEST,mbr.name))
        assert target.startswith(DEST+os.sep) or target==DEST, f'unsafe path {mbr.name}'
        assert mbr.isfile() or mbr.isdir(), f'unexpected member type {mbr.name}'
        top=mbr.name.split('/')[0]
        assert top in {'train','val','train_manifest.csv','val_manifest.csv','patient_splits.csv'}, f'unexpected (possible holdout) member {mbr.name}'
    t.extractall(DEST)
print('sha256 OK;', len(names), 'members; extracted to', DEST)

download: s3://ajay-mlp1-lgg-044450716277-use1/data/trainval/trainval.tar to ../.downloads/trainval.tar


/tmp/ipykernel_808/1276283656.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(DEST)


sha256 OK; 7009 members; extracted to /home/sagemaker-user/brain-mri-segmentation/data


In [6]:
import pandas as pd, hashlib
D=f'{ROOT}/data'; bad=0; n=0
assert not os.path.exists(f'{D}/test'), 'test data must not be here'
for s in ['train','val']:
    df=pd.read_csv(f'{D}/{s}_manifest.csv')
    for _,r in df.iterrows():
        for c,h in [('image_path','image_sha256'),('mask_path','mask_sha256')]:
            n+=1; bad+= hashlib.sha256(open(f'{D}/'+r[c],'rb').read()).hexdigest()!=r[h]
print('files checked',n,'mismatches',bad)

files checked 6722 mismatches 0


In [7]:
!cd ~/brain-mri-segmentation && python -m pytest -q tests

.........................                                                [100%]
25 passed in 4.29s
